## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        self.W1=tf.Variable(tf.random.truncated_normal([28*28,128],stddev=0.1))
        self.b1=tf.Variable(tf.zeros(128))
        self.W2=tf.Variable(tf.random.truncated_normal([128,10],stddev=0.1))
        self.b2=tf.Variable(tf.zeros(10))
        ####################
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        #将每张图拉平成784
        x=tf.reshape(x,[-1,784])

        #计算隐藏层输出
        #tf.matmul(x,self.w1): x的形状[batch,784] w1的形状[784,128] x*w1[batch,128]
        h=tf.nn.relu(tf.matmul(x,self.W1)+self.b1)

        #计算每个样本的1-10的分数，哪个最大预测成哪个数字
            #tf.matmul(h, self.W2)：把 128 维特征映射到 10 维类别空间。
            #+ self.b2：给每个类别再加一个可学习偏置。
            #得到 logits（形状 [batch, 10]）：每张图对应 10 个分数。
        logits=tf.matmul(h,self.W2)+self.b2
        ####################
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [5]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.4526322 ; accuracy 0.08476666
epoch 1 : loss 2.4341223 ; accuracy 0.090383336
epoch 2 : loss 2.4166334 ; accuracy 0.096316665
epoch 3 : loss 2.4000502 ; accuracy 0.10225
epoch 4 : loss 2.3842776 ; accuracy 0.10818333
epoch 5 : loss 2.3692346 ; accuracy 0.114133336
epoch 6 : loss 2.3548484 ; accuracy 0.1201
epoch 7 : loss 2.3410592 ; accuracy 0.12601666
epoch 8 : loss 2.3278162 ; accuracy 0.13185
epoch 9 : loss 2.3150718 ; accuracy 0.13795
epoch 10 : loss 2.302785 ; accuracy 0.1443
epoch 11 : loss 2.2909188 ; accuracy 0.1496
epoch 12 : loss 2.279439 ; accuracy 0.1554
epoch 13 : loss 2.2683187 ; accuracy 0.16093333
epoch 14 : loss 2.2575305 ; accuracy 0.16706666
epoch 15 : loss 2.2470508 ; accuracy 0.17311667
epoch 16 : loss 2.2368565 ; accuracy 0.17911667
epoch 17 : loss 2.2269304 ; accuracy 0.18496667
epoch 18 : loss 2.2172525 ; accuracy 0.19118333
epoch 19 : loss 2.2078075 ; accuracy 0.19665
epoch 20 : loss 2.1985788 ; accuracy 0.20231667
epoch 21 : loss 2.1895535 ; a